# Bronze Layer Data Load

This notebook moves data from the **raw** layer (source data) to the **bronze** layer (cleaned and tracked data).

---

## What Does This Notebook Do?

It reads tables from `retail.raw` schema and writes them to `retail.bronze` schema with tracking columns added.

---

## Two Ways to Handle Data

### 1. **SCD2** - Track Changes Over Time
- Used for: **Dimension tables** like customers and addresses
- What it does: Keeps history of all changes
- How it works:
  - When a record changes, the old version is marked as expired (`is_current = false`)
  - A new version is inserted with the updated data (`is_current = true`)
- Tracking columns added: `start_date`, `end_date`, `is_current`, `row_hash`

### 2. **APPEND** - Just Add New Records
- Used for: **Fact tables** like orders, payments, refunds
- What it does: Simply adds new records without tracking changes
- Tracking columns added: `created_at`, `updated_at`

---

## Two Ways to Load Data

### 1. **Full Load** - Read Everything
- Reads the entire source table every time
- Compares with existing bronze data
- Good for smaller tables

### 2. **Incremental Load** - Read Only New Records
- Uses a watermark (last loaded timestamp) to filter
- Only reads records created/updated since last run
- More efficient for large tables that update frequently
- Uses temporary views (`{table_name}_vw`) to filter data

---

## Control Table

The **load_control** table remembers the last time data was loaded:
- `table_name`: Which table we're tracking
- `wm_col`: Which column has the timestamp (usually `created_timestamp`)
- `last_loaded_wm`: Last timestamp we loaded
- `updated_at`: When we last ran the load

---

## How It Works

1. **Configuration**: Table settings are loaded from `bronze_config.json`
2. **For each table**:
   - Check if it's a full or incremental load
   - Apply SCD2 or APPEND strategy
   - Add tracking columns
   - Write to bronze layer
3. **Update control table** (for incremental loads only)

In [0]:
# %sql
# RESTORE TABLE retail.raw.customers TO VERSION AS OF 0;
# RESTORE TABLE retail.bronze.load_control TO VERSION AS OF 0;
# DROP TABLE IF EXISTS retail.bronze.customers;

In [0]:
control_table_name = "retail.bronze.load_control"

In [0]:

# Create control table if it doesn't exist
spark.sql(f"""
    CREATE OR REPLACE TABLE {control_table_name} (
        table_name STRING,
        wm_col STRING,
        last_loaded_wm TIMESTAMP,
        updated_at TIMESTAMP
    )
    USING DELTA
""")

# Initialize control record for customers table
spark.sql(f"""
    MERGE INTO {control_table_name} AS target
    USING (SELECT 'customers' as table_name) AS source
    ON target.table_name = source.table_name
    WHEN NOT MATCHED THEN
        INSERT (table_name,  wm_col, last_loaded_wm, updated_at)
        VALUES ('retail.raw.customers', 'created_timestamp',  
                timestamp('1900-01-01 00:00:00'), current_timestamp())
""")

print(f"Control table created/verified: {control_table_name}")
display(spark.table(control_table_name))

In [0]:
# Import required libraries
from pyspark.sql.functions import col, lit, current_timestamp, md5, concat_ws
from pyspark.sql.types import TimestampType
from delta.tables import DeltaTable
from functools import reduce
import time
from datetime import datetime

import json
with open('../../configs/bronze_config.json', 'r') as f:
    table_config = json.load(f)

print(f"Configuration loaded: {len(table_config)} tables")

In [0]:
def get_incremental_records(full_nm):
    view_name = f"{full_nm}_vw"
    
    # Drop view if exists and recreate
    spark.sql(f"DROP VIEW IF EXISTS {view_name}")
   
    # Get watermark column and last_loaded_wm for the table
    wm_info = spark.sql(f"""
        SELECT wm_col, last_loaded_wm 
        FROM {control_table_name} 
        WHERE table_name = '{full_nm}'
    """).collect()[0]
    wm_col = wm_info[0]
    last_loaded_wm = wm_info[1]
    
    # Create view that filters based on watermark column
    spark.sql(f"""
        CREATE VIEW {view_name} AS
        SELECT *
        FROM {full_nm}
        WHERE {wm_col} > timestamp('{last_loaded_wm}')
    """)
    
    print(f"Incremental view created: {view_name}")

In [0]:
def update_control_table(source_tbl_nm):
    # Get watermark column name from control table
    wm_col = spark.sql(f"""
        SELECT wm_col 
        FROM {control_table_name} 
        WHERE table_name = '{source_tbl_nm}'
    """).collect()[0][0]
    
    # Get max value of watermark column from source table
    max_timestamp = spark.table(source_tbl_nm).agg({wm_col: "max"}).collect()[0][0]
    
    # Update control table with new watermark
    spark.sql(f"""
        UPDATE {control_table_name}
        SET last_loaded_wm = timestamp('{max_timestamp}'),
            updated_at = current_timestamp()
        WHERE table_name = '{source_tbl_nm}'
    """)
    print(f"Updated watermark to: {max_timestamp} for column: {wm_col}")

In [0]:
def load_scd2(source_table, target_table, primary_keys, catalog_name, source_schema, target_schema, load_type, tracking_cols):
    try:
        # Determine source based on load_type
        full_nm = f"{catalog_name}.{source_schema}.{source_table}"
        if load_type == "incremental":
            get_incremental_records(full_nm)
            source_tbl_nm = f"{catalog_name}.{source_schema}.{source_table}_vw"
            print(f"Using incremental view: {source_tbl_nm}")
        else:
            source_tbl_nm = f"{catalog_name}.{source_schema}.{source_table}"
            print(f"Using full table: {source_tbl_nm}")
        
        target_tbl_nm = f"{catalog_name}.{target_schema}.{target_table}"
        
        # Read source data and fill nulls
        source_df = spark.table(source_tbl_nm) \
            .where(reduce(lambda a, b: a & b, [col(pk).isNotNull() for pk in primary_keys])) \
            .orderBy([col(c).asc_nulls_last() for c in spark.table(source_tbl_nm).columns]) \
            .dropDuplicates(primary_keys) \
            .fillna("N/A")
        
        # Check if there's any data to process
        source_count = source_df.count()
       
        # Check if target table exists
        target_exists = spark.catalog.tableExists(target_tbl_nm)
        tracking_col_list = [c for c in tracking_cols.split(',')]
        
        if not target_exists:
            # Initial load: add audit and SCD2 columns
            bronze_df = source_df \
                .withColumn(
                "row_hash",   md5(concat_ws("|", *[col(c).cast("string") for c in tracking_col_list]))) \
                .withColumn("created_at", current_timestamp()) \
                .withColumn("updated_at", current_timestamp()) \
                .withColumn("start_date", current_timestamp()) \
                .withColumn("end_date", lit(None).cast(TimestampType())) \
                .withColumn("is_current", lit(True))
            
            bronze_df.write.mode("overwrite").saveAsTable(target_tbl_nm)
            
            # Update watermark if incremental
            if load_type == "incremental":
                update_control_table(full_nm)
            return {"status": "success","operation": "initial_load","records": bronze_df.count()}
        
        else:
            # Incremental load: perform SCD2 merge
            
            # CRITICAL: Exclude audit columns from hash comparison
            audit_columns = ['create_timestamp','created_at', 'updated_at', 'start_date', 'end_date', 'is_current']
            source_cols = [c for c in source_df.columns if c not in audit_columns]

            # Calculate hash for source (using only tracking columns)
            source_with_hash = source_df.withColumn(
                "row_hash", 
                md5(concat_ws("|", *[col(c).cast("string") for c in tracking_col_list]))
            ).withColumn("created_at", current_timestamp()) \
             .withColumn("updated_at", current_timestamp()) \
             .withColumn("start_date", current_timestamp()) \
             .withColumn("end_date", lit(None).cast(TimestampType())) \
             .withColumn("is_current", lit(True))
            
            # Build primary key join condition for MERGE
            pk_condition = " AND ".join([f"source.{pk} = target.{pk}" for pk in primary_keys])
            
            # Get Delta table
            delta_table = DeltaTable.forName(spark, target_tbl_nm)
            
            # STEP 1: MERGE to expire old records and insert updated versions
            # Match on PK where hash differs AND record is current
            print("Step 1: Expiring old records and inserting updated versions...")
            
            merge_result = delta_table.alias("target") \
                .merge(
                    source_with_hash.alias("source"),
                    f"{pk_condition} AND target.is_current = true"
                ) \
                .whenMatchedUpdate(
                    condition="target.row_hash != source.row_hash",
                    set={
                        "is_current": "false",
                        "end_date": "current_timestamp()",
                        "updated_at": "current_timestamp()"
                    }
                ) \
                .whenNotMatchedInsertAll() \
                .execute()
            
            # STEP 2: LEFT ANTI JOIN to find records that were expired (changed)
            # These need new versions inserted
            print("Step 2: Inserting new versions of changed records...")
            
            target_current = spark.table(target_tbl_nm).filter(col("is_current") == True)
            
            # Build join condition for LEFT ANTI JOIN
            pk_join_condition = reduce(lambda a, b: a & b, 
                                      [col(f"source.{pk}") == col(f"target.{pk}") for pk in primary_keys])
            
            # Find records in source that don't have a current version in target
            # These are records that were just expired in STEP 1
            changed_records = source_with_hash.alias("source") \
                .join(
                    target_current.alias("target"),
                    pk_join_condition,
                    "left_anti"
                )
            
            changed_count = changed_records.count()
            
            if changed_count > 0:
                print(f"Inserting {changed_count} new versions of changed records")
                changed_records.write.mode("append").saveAsTable(target_tbl_nm)
            
            # Update watermark if incremental
            if load_type == "incremental":
                update_control_table(full_nm)

            return {
                "status": "success",
                "operation": "incremental_load",
                "records_processed": source_count,
                "changed_records": changed_count
            }
    
    except Exception as e:
        return {"status": "failed", "error": str(e)}

print("Optimized SCD2 load function loaded")

In [0]:

def load_append(source_table, target_table, primary_keys, catalog_name, source_schema, target_schema):
    try:
        # Build fully qualified table names
        source_table_name = f"{catalog_name}.{source_schema}.{source_table}"
        target_table_name = f"{catalog_name}.{target_schema}.{target_table}"
        
        # Read source data
        source_df = spark.table(source_table_name)
        
        # Add audit columns and append to target
        bronze_df = source_df \
            .withColumn("created_at", current_timestamp()) \
            .withColumn("updated_at", current_timestamp())
        
        bronze_df.write.mode("append").saveAsTable(target_table_name)

        return {
            "status": "success",
            "operation": "append",
            "new_records": source_df.count()
        }
    
    except Exception as e:
        return {"status": "failed", "error": str(e)}

print("Append load function loaded")

## Main Processing

This section processes all tables defined in the configuration.

For each table:
1. Reads from source (raw layer)
2. Applies the configured load strategy
3. Writes to target (bronze layer)
4. Logs the results

In [0]:
def run():    # Process all tables 
    print(f"\n{'='*80}\nBronze Layer Load Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n{'='*80}")

    results = []
    control_table_name = f"retail.bronze.load_control"

    for config in table_config:
        table_name = config['table_name']
        primary_keys = config['primary_keys']
        load_strategy = config['load_strategy']
        load_type = config.get('load_type', 'full')  # Default to full if not specified
        catalog_name = config['catalog_name']
        source_schema = config['source_schema']
        target_schema = config['target_schema']
        print(f"\nProcessing: {table_name}")
        print(f"Strategy: {load_strategy}")
        print(f"Load Type: {load_type}")
        if load_type == "incremental":
            print(f"From: {catalog_name}.{source_schema}.{table_name}_vw")
        else:
            print(f"From: {catalog_name}.{source_schema}.{table_name}")
        print(f"To: {catalog_name}.{target_schema}.{table_name}")
        
        try:
            # Call the appropriate load function
            if load_strategy == "SCD2":
                result = load_scd2(
                                source_table=table_name,
                                target_table=table_name,
                                primary_keys=primary_keys,
                                catalog_name=catalog_name,
                                source_schema=source_schema,
                                target_schema=target_schema,
                                load_type=load_type,
                                tracking_cols=config['tracking_cols']
                                )
            elif load_strategy == "APPEND":
                result = load_append(
                                source_table=table_name,
                                target_table=table_name,
                                primary_keys=primary_keys,
                                catalog_name=catalog_name,
                                source_schema=source_schema,
                                target_schema=target_schema
                                        )
            else:
                result = {"status": "failed", "error": f"Unknown strategy: {load_strategy}"}
            
            # Print result
            if result['status'] == 'success':
                if result['operation'] == 'initial_load':
                    print(f"Result: SUCCESS - Initial load: {result.get('records', 0)} records")
                elif result['operation'] == 'incremental_load':
                    print(f"Result: SUCCESS - New: {result.get('new_records', 0)}, Changed: {result.get('changed_records', 0)}")
                elif result['operation'] == 'no_new_data':
                    print(f"Result: SUCCESS - No new data to process")
                else:
                    print(f"Result: SUCCESS - Appended: {result.get('new_records', 0)} records")
            else:
                print(f"Result: FAILED - {result.get('error', 'Unknown error')}")
            
            results.append(result)
        
        except Exception as e:
            print(f"Result: FAILED - {str(e)}")
            results.append({"status": "failed", "error": str(e)})

    print(f"\n{'='*80}\nBronze Layer Load Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n{'='*80}")

## Running Bronze Load

In [0]:
# Run the flow
run()

In [0]:
bronze_tables = [
    "retail.bronze.customers",
    "retail.bronze.addresses",
    "retail.bronze.orders",
    "retail.bronze.payments",
    "retail.bronze.refunds"
]

for tbl in bronze_tables:
    print(f"\nDisplaying records from: {tbl}")
    display(spark.table(tbl).limit(100))

In [0]:
%sql
table retail.raw.customers

In [0]:
%sql


INSERT INTO retail.raw.customers (customer_id, customer_name, date_of_birth, telephone, email, member_since, created_timestamp)
VALUES 
   (9179, 'Richard Cox', '1996-10-25', '+1 6680703335', 'richardcox_v555555@example.com', '2026-03-31', current_timestamp()),
   (4858, 'Carla Morton', '2004-06-21', '+1 9999999999', 'joseph88@mail.com', '2024-09-15', current_timestamp()),
   (6973, 'Tracy Cole', '1998-07-24', '+1 5515836612', 'tony46@mail.com', '2024-12-16',  current_timestamp())


In [0]:
%sql
select * from retail.bronze.customers where customer_id IN (9179,4858,6973);